In [8]:
from dotenv import load_dotenv
import os
from espn_api.football import League

load_dotenv()

league_id = os.getenv('LEAGUE_ID')
swid = os.getenv('SWID')
espn_s2 = os.getenv('ESPN_S2')
year = 2024
week = 1

league = League(league_id=league_id, year=year, espn_s2=espn_s2, swid=swid)


In [9]:
from dotenv import load_dotenv
import os
from supabase import create_client, Client
import json


def get_supabase_client():
    # load supabase client    
    supabase: Client = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))
    return supabase

# supabase helper functions
def get_league_id_from_espn_league_id(supabase_client, external_league_id):
    response = supabase_client.table("leagues").select("id").eq("external_league_id", external_league_id).execute()
    return response.data[0]["id"]

def get_external_team_id_to_team_id_map(supabase_client, league_id):
    response = supabase_client.table("teams").select("id", "espn_team_id").eq("league_id", league_id).execute()
    team_map = {}
    for team in response.data:
        team_map[team["espn_team_id"]] = team["id"]
    return team_map

def get_external_player_id_to_player_id_map(supabase_client):
    response = supabase_client.table("nfl_players").select("id, espn_player_id").execute()
    player_map = {}
    for player in response.data:
        player_map[player["espn_player_id"]] = player["id"] 
    return player_map

def get_game_id_for_team_in_week(supabase_client, team_id, week, year):
    response = supabase_client.table("nfl_schedule").select("id").or_(f"home_team_id.eq.{team_id},away_team_id.eq.{team_id}").eq("week", week).eq("year", year).execute()
    
    if len(response.data) == 0:
        return None
    
    return response.data[0]["id"]

def get_team_abbrev_to_team_id_map(supabase_client):
    response = supabase_client.table("nfl_teams").select("id, team_abbrev").execute()
    team_map = {}
    for team in response.data:
        team_map[team["team_abbrev"]] = team["id"]
    return team_map

def get_player_id_to_team_id_map(supabase_client):
    response = supabase_client.table("nfl_players").select("id, team_id").execute()
    player_map = {}
    for player in response.data:
        player_map[player["id"]] = player["team_id"]
    return player_map

def get_nfl_data_player_id_to_db_player_id_map(supabase_client, nfl_data_player_id_to_espn_id_map):
    response = supabase_client.table("nfl_players").select("id, espn_player_id").execute()
    
    espn_player_id_to_id_map = {}
    nfl_data_player_id_to_id_map = {}
    for player in response.data:
        espn_player_id_to_id_map[str(player["espn_player_id"])] = player["id"]
    
    for player_id in nfl_data_player_id_to_espn_id_map:
        espn_player_id = nfl_data_player_id_to_espn_id_map[player_id]
        if espn_player_id in espn_player_id_to_id_map:
            nfl_data_player_id_to_id_map[player_id] = espn_player_id_to_id_map[espn_player_id]
        # else:
        #     print(f"Player {espn_player_id} not found in nfl_data_player_id_to_espn_id_map")
    
    return nfl_data_player_id_to_id_map
    
def get_teams_in_league(supabase_client, league_id):
    response = supabase_client.table("teams").select("id, team_name, espn_team_id").eq("league_id", league_id).execute()
    return response.data

def get_espn_player_id_to_player_id_map(supabase_client):
    response = supabase_client.table("nfl_players").select("id, espn_player_id").execute()
    player_map = {}
    for player in response.data:
        player_map[player["espn_player_id"]] = player["id"]
    return player_map

def get_current_player_id_to_team_id_roster_map(supabase_client, league_id, year):
    response = supabase_client.table("fantasy_team_roster").select("player_id, team_id").eq("league_id", league_id).eq("year", year).execute()
    player_map = {}
    for player in response.data:
        player_map[player["player_id"]] = player["team_id"]
    return player_map

In [14]:
from datetime import datetime, timezone


def get_and_insert_current_scores(league, week, supabase_client):
    current_scores = []
    db_league_id = get_league_id_from_espn_league_id(supabase_client, league.league_id)
    espn_team_id_to_team_id_map = get_external_team_id_to_team_id_map(supabase_client, db_league_id)
    for box_score in league.box_scores(week):
        home_team_id = box_score.home_team.team_id
        away_team_id = box_score.away_team.team_id
        
        db_home_team_id = espn_team_id_to_team_id_map[home_team_id]
        db_away_team_id = espn_team_id_to_team_id_map[away_team_id]
        
        home_team_points = box_score.home_score
        away_team_points = box_score.away_score
        
        current_scores.append({
            "league_id": db_league_id,
            "team_id": db_home_team_id,
            "points": home_team_points,
            "week": week,
            "year": league.year,
            "created_at": datetime.now(timezone.utc).isoformat()
        })
        
        current_scores.append({
            "league_id": db_league_id,
            "team_id": db_away_team_id,
            "points": away_team_points,
            "week": week,
            "year": league.year,
            "created_at": datetime.now(timezone.utc).isoformat()
        })
        
    supabase_client.table("fantasy_team_points_live_tracking").insert(current_scores).execute()
    return current_scores

current_scores = get_and_insert_current_scores(league, week, supabase_client=get_supabase_client())
print(current_scores)

[{'league_id': 'b63aa425-14ec-482c-b834-0d4f820c561c', 'team_id': '14af6e91-3be7-4a51-9efb-a2afff769718', 'points': 110.34, 'week': 1, 'year': 2024, 'created_at': '2025-08-27T02:20:36.039853+00:00'}, {'league_id': 'b63aa425-14ec-482c-b834-0d4f820c561c', 'team_id': 'd7b3bb3a-190f-4407-b43c-02f3cbb36710', 'points': 111.82, 'week': 1, 'year': 2024, 'created_at': '2025-08-27T02:20:36.039906+00:00'}, {'league_id': 'b63aa425-14ec-482c-b834-0d4f820c561c', 'team_id': 'f7fdb028-1ae4-4c7d-adec-7d4c5095a124', 'points': 150.68, 'week': 1, 'year': 2024, 'created_at': '2025-08-27T02:20:36.039919+00:00'}, {'league_id': 'b63aa425-14ec-482c-b834-0d4f820c561c', 'team_id': '6f19fba8-bd44-4e29-a664-388b48e5b88a', 'points': 108.98, 'week': 1, 'year': 2024, 'created_at': '2025-08-27T02:20:36.039924+00:00'}, {'league_id': 'b63aa425-14ec-482c-b834-0d4f820c561c', 'team_id': 'f3ad7f4a-3429-4580-be90-046edf970fba', 'points': 84.46, 'week': 1, 'year': 2024, 'created_at': '2025-08-27T02:20:36.039932+00:00'}, {'lea